In [ ]:
# ============================================================
# Ubermag Uniform OOP Simulation — Local Windows
# Fully out-of-plane uniform magnetisation
# Output: Bz_NV_uniform_oop.npy, Mz_true_uniform_oop.npy
# Compatible with Ubermag_Model_Verification pipeline
# ============================================================
# Author: Yue Yu, TU Dresden / ct.qmat
# ============================================================

# ── Cell 1: Imports ──────────────────────────────────────────
import discretisedfield as df
import micromagneticmodel as mm
import oommfc as oc
import numpy as np
import matplotlib.pyplot as plt
import json, os
from datetime import datetime

# ── Cell 2: Paths ────────────────────────────────────────────
CASE     = "uniform_oop"
base_dir = r"C:\Users\Jianpeng Chen\Desktop\TUD 2025\MR\Ubermag"

# 日期序号保存目录，和 neel_dw / skyrmion 风格一致
# Dated output directory, consistent with neel_dw / skyrmion
date_str = datetime.now().strftime("%Y%m%d")
idx = 1
while os.path.exists(os.path.join(base_dir, CASE, f"{date_str}_{idx:02d}")):
    idx += 1

case_dir = os.path.join(base_dir, CASE, f"{date_str}_{idx:02d}")
os.makedirs(case_dir, exist_ok=True)
print("Save directory:", case_dir)

# ── Cell 3: Geometry ─────────────────────────────────────────
Lx   = 256 * 5e-9   # 1280 nm，与 neel_dw / skyrmion 一致
Ly   = 256 * 5e-9   # 1280 nm
Lz   = 5e-9         # 单层厚度 / Single layer thickness
cell = (5e-9, 5e-9, 5e-9)
z_NV = 50e-9        # NV 离样品高度 / NV standoff height

mesh = df.Mesh(
    p1=(-Lx/2, -Ly/2, 0),
    p2=( Lx/2,  Ly/2, Lz),
    cell=cell
)
print(f"Mesh: {mesh.n} cells  |  pixel size: {cell[0]*1e9:.0f} nm")
print(f"Domain size: {Lx*1e9:.0f} x {Ly*1e9:.0f} nm")

# ── Cell 4: Material parameters ──────────────────────────────
Ms = 8e5    # A/m  — 饱和磁化强度 / Saturation magnetisation
A  = 1e-11  # J/m  — 交换刚度 / Exchange stiffness
# 均匀 OOP 不需要 DMI 和各向异性
# Uniform OOP does not require DMI or anisotropy

# ── Cell 5: Initial magnetisation — uniform OOP ──────────────
# 所有自旋均匀朝 +z 方向 / All spins uniformly pointing +z
def uniform_oop(pos):
    return (0, 0, Ms)

# ── Cell 6: Build system & relax ─────────────────────────────
system = mm.System(name="uniform_oop")
system.energy = (
    mm.Exchange(A=A)
    + mm.Demag()
    # 无 DMI，无各向异性 / No DMI, no anisotropy
)
system.m = df.Field(mesh, nvdim=3, value=uniform_oop, norm=Ms)

md = oc.MinDriver()
md.drive(system)
print("Relaxation done.")

# ── Cell 7: Extract Mx, My, Mz ───────────────────────────────
m_sel = system.m.sel(z=Lz / 2)
Mx = np.squeeze(m_sel.x.array)
My = np.squeeze(m_sel.y.array)
Mz = np.squeeze(m_sel.z.array)
nx, ny = Mz.shape
print(f"Mz range: {Mz.min():.1f} ~ {Mz.max():.1f} A/m")
print(f"Array shape: {Mz.shape}")

# ── Cell 8: Plot Mz ──────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

im0 = ax.imshow(
    Mz / Ms,
    extent=[-Lx/2*1e6, Lx/2*1e6, -Ly/2*1e6, Ly/2*1e6],
    origin="lower", cmap="RdBu_r", vmin=-1, vmax=1
)
plt.colorbar(im0, ax=ax, label="Mz / Ms")
ax.set_title("Mz — Uniform OOP (relaxed)")
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")

plt.tight_layout()
plt.savefig(os.path.join(case_dir, "Mz_uniform_oop.png"), dpi=150)
plt.show()

# ── Cell 9: Compute Bz stray field ───────────────────────────
# FFT 传播公式，与 neel_dw / skyrmion 符号一致
# FFT propagation formula, consistent sign convention with neel_dw / skyrmion
dx = cell[0]
dy = cell[1]
kx_arr  = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
ky_arr  = 2 * np.pi * np.fft.fftfreq(ny, d=dy)
kx_grid, ky_grid = np.meshgrid(kx_arr, ky_arr, indexing="ij")
k = np.sqrt(kx_grid**2 + ky_grid**2)
k[0, 0] = 1e-12   # 避免 DC 奇点 / Avoid DC singularity

Mx_k = np.fft.fft2(Mx)
My_k = np.fft.fft2(My)
Mz_k = np.fft.fft2(Mz)

mu0  = 4 * np.pi * 1e-7
Bz_k = mu0 / (2 * k) * (
    -1j * kx_grid * Mx_k    # 负号，与 neel_dw 一致 / Negative sign, consistent with neel_dw
    - 1j * ky_grid * My_k
    + k * Mz_k
)

# 传播到 NV 高度 / Propagate to NV height
Bz_NV = np.real(np.fft.ifft2(Bz_k * np.exp(-k * z_NV)))

print(f"Bz range at {z_NV*1e9:.0f} nm: "
      f"{Bz_NV.min()*1e3:.3f} ~ {Bz_NV.max()*1e3:.3f} mT")
print(f"Bz mean: {Bz_NV.mean()*1e3:.3f} mT")
print(f"Bz at center: {Bz_NV[nx//2, ny//2]*1e3:.3f} mT")
print(f"Bz at edge:   {Bz_NV[0, 0]*1e3:.3f} mT")

# ── Cell 10: Plot Bz ─────────────────────────────────────────
plt.figure(figsize=(6, 5))
im = plt.imshow(
    Bz_NV * 1e3,
    extent=[-Lx/2*1e6, Lx/2*1e6, -Ly/2*1e6, Ly/2*1e6],
    origin="lower", cmap="RdBu_r",
)
plt.colorbar(im, label="Bz at 50 nm (mT)")
plt.xlabel("x (µm)")
plt.ylabel("y (µm)")
plt.title("Stray Field Bz — Uniform OOP")
plt.tight_layout()
plt.savefig(os.path.join(case_dir, "Bz_uniform_oop.png"), dpi=150)
plt.show()

# ── Cell 11: Save .npy ───────────────────────────────────────
np.save(os.path.join(case_dir, "Bz_NV_uniform_oop.npy"), Bz_NV)
np.save(os.path.join(case_dir, "Mz_true_uniform_oop.npy"), Mz)
print("Saved:")
print(f"  {case_dir}/Bz_NV_uniform_oop.npy   shape={Bz_NV.shape}")
print(f"  {case_dir}/Mz_true_uniform_oop.npy  shape={Mz.shape}")

# ── Cell 12: Save JSON ───────────────────────────────────────
synthetic_data = {
    "MagneticFieldImage (2D double array)": Bz_NV.tolist(),
    "PixleSizeX (double)": dx,
    "PixleSizeY (double)": dy,
    "NVTheta (double)": 0,
    "NVPhi (double)": 0,
    "NVHeight (double)": z_NV,
    "MagnetisationType (string)": "OOP",
    "SampleIdentification (string)": "Uniform_OOP",
    "MeasurementIdentification (string)": "Synthetic_Test",
    "MeasurementApparatus (string)": "Simulation"
}
with open(os.path.join(case_dir, "uniform_oop.json"), "w") as f:
    json.dump(synthetic_data, f)
print("JSON saved.")

# ── Cell 13: Pipeline integration check ──────────────────────
Bz_check = np.load(os.path.join(case_dir, "Bz_NV_uniform_oop.npy"))
Mz_check = np.load(os.path.join(case_dir, "Mz_true_uniform_oop.npy"))
print("\n=== Pipeline integration check ===")
print(f"Bz_NV shape : {Bz_check.shape}  (expected (256, 256))")
print(f"Mz_true shape: {Mz_check.shape}  (expected (256, 256))")
print(f"Bz_NV dtype : {Bz_check.dtype}")
print(f"Mz_true dtype: {Mz_check.dtype}")
print(f"\nReplace in your Notebook Cell 2:")
print(f'  Bz_NV  = np.load(r"{case_dir}\\Bz_NV_uniform_oop.npy")')
print(f'  Mz_true = np.load(r"{case_dir}\\Mz_true_uniform_oop.npy")')